# Explore the mesh - Phases 1-4 & 6

Loading, visualising, validating, normalising and measuring the symmetry of a
real ARKit capture.

**This notebook needs a real capture in `data/raw/`.** It generates no data. If a
cell reports "no captures found", export one from the iOS app first.

Read `docs/ASSUMPTIONS.md` alongside this - it lists what to verify here.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # run from notebooks/

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

## 0. What is actually in the file?

Assumes nothing about the schema. Run this before anything else.

In [ ]:
from src.data_loader import find_capture_files
from src.config import RAW_DIR

paths = find_capture_files(RAW_DIR)
print(f"{len(paths)} capture(s) found under {RAW_DIR}")
for p in paths[:10]:
    print("  ", p.relative_to(RAW_DIR))
assert paths, "No captures. Export from the iOS app into data/raw/<participant_id>/ first."
CAPTURE = paths[0]

In [ ]:
from src.inspect_schema import inspect
from src.config import SchemaMap

inspect(CAPTURE, SchemaMap.load())

## 1. Load it

Any field the loader had to infer appears under `Assumptions:` - read that line
every time. Nothing is guessed silently.

In [ ]:
from src.data_loader import load_capture

capture = load_capture(CAPTURE)
print(capture.summary())
print()
print("vertices: ", capture.vertices.shape, capture.vertices.dtype)
print("triangles:", capture.triangles.shape, capture.triangles.dtype)
print("eye transforms:", capture.has_eye_transforms())

## 2. Look at it

The single most important verification step in the whole pipeline: does the
exported geometry actually look like the face you captured?

Opens an interactive window (Open3D if installed, otherwise matplotlib).

In [ ]:
from src.visualization import show_mesh, open3d_available
print("Open3D available:", open3d_available(), "(matplotlib fallback otherwise)")

show_mesh(capture.vertices, capture.triangles, title=f"{capture.key} - raw export")

In [ ]:
# Wireframe: confirms the triangle topology, not just the point positions.
show_mesh(capture.vertices, capture.triangles, wireframe=True, title="topology")

# Point cloud: ignores topology entirely.
show_mesh(capture.vertices, None, title="vertices only")

## 3. Validate

Per-capture integrity plus cohort-wide consistency.

In [ ]:
from src.validation import validate_capture, validate_cohort

print(validate_capture(capture, CAPTURE).render())

In [ ]:
cohort = validate_cohort(RAW_DIR)
print(f"PASS {cohort.n_pass}   WARN {cohort.n_warn}   FAIL {cohort.n_fail}")
print(f"Cohort topology (derived, not assumed): {cohort.expected_vertices} vertices / "
      f"{cohort.expected_triangles} triangles")
for note in cohort.notes:
    print("  *", note)

## 4. Detect the axis convention

Detected from your real data with reported evidence - see
`docs/NORMALIZATION.md` for the rules. **Check `left_right_margin`:** below `2x`
means the detection was not decisive.

In [ ]:
import json
from src.normalization import detect_axis_convention
from src.data_loader import load_all

captures = load_all(RAW_DIR)
convention = detect_axis_convention(captures)
print(convention.describe())
print()
print(json.dumps(convention.evidence, indent=2))

### Confirm it visually

Green `+Y` must point at the forehead, blue `+Z` out of the face. Only save with
`--confirm` once you have actually looked.

In [ ]:
from src.normalization import normalize_capture

face = normalize_capture(capture, convention)
show_mesh(face.vertices, face.triangles, show_axes=True,
          title=f"{face.key} - canonical frame (+X left, +Y up, +Z out)")

In [ ]:
print("origin mode      :", face.origin_mode)
print("origin offset (m):", np.round(face.origin_offset_m, 5))
print("scale mode       :", face.scale_mode)
print("scale reference  :", f"{face.scale_reference_m:.6f} m")
print("side resolved    :", face.side_resolved)
for n in face.notes:
    print("  note:", n)

### Does normalisation actually work?

The real test: **two captures of the same person at different distances/angles.**
Ratio features should agree closely. If they do not, suspect assumption A4
(vertices exported in world space without being declared as such).

In [ ]:
from collections import Counter

counts = Counter(c.participant_id for c in captures)
repeated = [p for p, n in counts.items() if n > 1]

if not repeated:
    print("No participant has 2+ captures yet - capture a pair to run this check.")
else:
    pid = repeated[0]
    rows = []
    for c in captures:
        if c.participant_id != pid:
            continue
        f = normalize_capture(c, convention)
        lo, hi = f.vertices.min(0), f.vertices.max(0)
        rows.append({"capture": c.capture_id,
                     "scale_ref_m": f.scale_reference_m,
                     "width/height": (hi - lo)[0] / (hi - lo)[1],
                     "depth/height": (hi - lo)[2] / (hi - lo)[1]})
    df = pd.DataFrame(rows)
    display(df)
    print("\nSpread across captures of the same person (should be small):")
    print(df[["width/height", "depth/height"]].std())

## 5. Symmetry

The midplane is **fitted**, not assumed to be `x=0`. Default matching is
`point_to_surface`, which has no discretization floor - see the
`src/symmetry.py` docstring.

In [ ]:
from src.symmetry import measure_symmetry

result = measure_symmetry(face.vertices, face.triangles)
print(result.summary())
for n in result.notes:
    print("  note:", n)

In [ ]:
from src.visualization import symmetry_colors

show_mesh(face.vertices, face.triangles, colors=symmetry_colors(result.per_vertex),
          title=f"asymmetry - rms={result.symmetry_rms:.4f} (red = least symmetric)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(result.per_vertex, bins=60, color="#4a7ba7")
ax.axvline(result.symmetry_rms, color="crimson", label=f"rms = {result.symmetry_rms:.4f}")
ax.set_xlabel("per-vertex asymmetry (normalised by centroid size)")
ax.set_ylabel("vertices")
ax.legend()
ax.set_title("Distribution of asymmetry")
plt.tight_layout(); plt.show()

---
## Next

`explore_features.ipynb` - measurements, profiles, and the cohort table.